# Example processing script: Heart rate and acceleration

## Setup

See installation instructions [here](https://github.com/beatlab-mcmaster/BEATmonitor/blob/main/src/analysis/README.md)

### Set configuration file

In [1]:
CONFIG_FILE = "default.yml" # In this notebook, set the configuration file here

### Import libraries

In [2]:
import re  # Text search tools
from pathlib import Path  # noqa: F401 -- Path syntax

import holoviews as hv
from holoviews import dim, opts
import hvplot.pandas  # noqa: F401 -- Pandas + holoviews
import pandas as pd  # Data handling
import pytz  # Timezone tools

# Import beatwatch processing tools
from beatwatch_process.parsers import Parser, summarise_metadata, select_period
from beatwatch_process.process import upsample
from beatwatch_process.utils import get_valid_watch_files, load_config

Note: Logging from `beatwatch_process` library will be shown inline and the `__logs` folder.

### Configure analysis

In [3]:
cfg = load_config(CONFIG_FILE)
hz = round(1000 / cfg["rate_downsample"])  # TODO: add to beatwatch:utils

tz = pytz.timezone(cfg["timezone"])  # Initialize timezone

parser = Parser(cfg["timezone"]) # Initialize parser

2026-03-23 12:48:14 | INFO     | beatwatch_process.utils:load_config — Input path: raw: example_data
2026-03-23 12:48:14 | INFO     | beatwatch_process.utils:load_config — Output path: figures: results/443192324/figures
2026-03-23 12:48:14 | INFO     | beatwatch_process.utils:load_config — Output path: tables: results/443192324/tables
2026-03-23 12:48:14 | INFO     | beatwatch_process.utils:load_config — Output path: __cache: results/443192324/__cache
2026-03-23 12:48:14 | INFO     | beatwatch_process.utils:load_config — Output path: summary: results/443192324/summary


When the configuration is loaded, a `results` folder is created. Output from 
`beatwatch_process` functions are located in subfolders named by script.

TODO: fix subfolder names for jupyter notebook files
TODO: log other configuration variables, parser created

## Read data

### Search for valid files

In [4]:
f_data = get_valid_watch_files(cfg["paths_in"]["raw"])

2026-03-23 12:48:15 | SUCCESS  | beatwatch_process.utils:get_valid_watch_files — Found 3 valid files in: example_data


In [5]:
# See file names
for f in f_data:
    print(f)

10-09_time_14-38-58_2cee_W010.csv
10-09_time_14-38-58_0510_W002.csv
10-09_time_14-38-58_7eed_W004.csv


### Parse files

In [6]:
data = {} # Create a dictionary to hold individual file data.

# Read/parse each file
for f in f_data:
    data[f] = parser.parse_file(cfg["paths_in"]["raw"] / f)

2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:parse_file — Reading example_data/10-09_time_14-38-58_2cee_W010.csv
2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:parse_file — Reading example_data/10-09_time_14-38-58_0510_W002.csv
2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:parse_file — Reading example_data/10-09_time_14-38-58_7eed_W004.csv


### File metadata summary

In [7]:
metadata = summarise_metadata(data)
metadata.to_csv(cfg["paths_out"]["summary"] / "parsed_metadata.csv")

The summarised metadata is saved to the `results` folder.
We can also view it here:

In [8]:
metadata

,Parsed_on,StudyName,StudyInstance,Name,Serial,MAC,PhysicalID,start_State,start_DateTime,start_UNIXTimeStamp,...,stop_DateTime,stop_UNIXTimeStamp,stop_BatteryLife,stop_FreeStorage,stop_SamplesWritten,n_samples_hr,n_samples_accel,n_survey_responses,duration_hr,duration_accel
file_name,,,,,,,,,,,,,,,,,,,,,
10-09_time_14-38-58_2cee_W010.csv,2026-03-23T16:48:15.408776+00:00,NA,NA,10-09T14:38:58_2cee_W010,7440756e-c4ef9af8,e6:3d:7e:d8:2c:ee,W010,START_RECORD,Thu Oct 9 2025 09:38:58 GMT-0500,2025-10-09T14:38:58.714Z,...,Thu Oct 9 2025 10:33:49 GMT-0500,2025-10-09T15:33:49.455Z,68,5436860,"{'hrm': 70440, 'accel': 36762}",70440,36762,0,0 days 00:54:50.734000,0 days 00:54:50.680000
10-09_time_14-38-58_0510_W002.csv,2026-03-23T16:48:15.650456+00:00,NA,NA,10-09T14:38:58_0510_W002,6cc37388-a01c03b0,ee:22:5c:2e:05:10,W002,START_RECORD,Thu Oct 9 2025 10:38:58 GMT-0400,2025-10-09T14:38:58.215Z,...,Thu Oct 9 2025 11:33:47 GMT-0400,2025-10-09T15:33:47.748Z,72,20912,"{'hrm': 69288, 'accel': 37303}",69288,37303,0,0 days 00:54:49.526000,0 days 00:54:49.519000
10-09_time_14-38-58_7eed_W004.csv,2026-03-23T16:48:15.940162+00:00,NA,NA,10-09T14:38:58_7eed_W004,d408d083-1d653f7c,d6:a5:0f:e3:7e:ed,W004,START_RECORD,Thu Oct 9 2025 10:38:58 GMT-0400,2025-10-09T14:38:58.508Z,...,Thu Oct 9 2025 11:33:48 GMT-0400,2025-10-09T15:33:48.329Z,73,5354940,"{'hrm': 69855, 'accel': 36581}",69855,36581,0,0 days 00:54:49.812000,0 days 00:54:49.795000


### Parsed result

`data` is a dictionary of filenames containing metadata and dataframes for each data type:

To iterate through data, use `data.items()`

In [9]:
for k, v in data.items():
    print(f'Key: {k}') # The raw data file name is the key
    for i in v:
        print(f' item: {i}') # Individual (parsed) dataframes associated with each file

Key: 10-09_time_14-38-58_2cee_W010.csv
 item: metadata
 item: data_hr
 item: data_accel
Key: 10-09_time_14-38-58_0510_W002.csv
 item: metadata
 item: data_hr
 item: data_accel
Key: 10-09_time_14-38-58_7eed_W004.csv
 item: metadata
 item: data_hr
 item: data_accel


### Structure of parsed dataframes

#### Heart rate data

In [10]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_hr"].head()

,time_elapsed,heart_rate_bpm,confidence,ppg_raw,ppg_filter,time_absolute
0,0 days 00:00:14.472000,77,81,6486,6144,2025-10-09 10:39:13.186000-04:00
1,0 days 00:00:14.521000,77,81,6462,-768,2025-10-09 10:39:13.235000-04:00
2,0 days 00:00:14.595000,77,81,6462,-256,2025-10-09 10:39:13.309000-04:00
3,0 days 00:00:14.636000,77,81,6478,4096,2025-10-09 10:39:13.350000-04:00
4,0 days 00:00:14.682000,77,81,6462,-512,2025-10-09 10:39:13.396000-04:00


#### Acceleration data

In [11]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_accel"].head()

,time_elapsed,x,y,z,magnitude,difference,time_absolute
0,0 days 00:00:00.113000,-60,-22,-984,986,3,2025-10-09 10:38:58.827000-04:00
1,0 days 00:00:00.198000,-59,-21,-988,990,4,2025-10-09 10:38:58.912000-04:00
2,0 days 00:00:00.272000,-61,-19,-991,994,4,2025-10-09 10:38:58.986000-04:00
3,0 days 00:00:00.352000,-63,-21,-994,996,3,2025-10-09 10:38:59.066000-04:00
4,0 days 00:00:00.432000,-66,-22,-997,1000,5,2025-10-09 10:38:59.146000-04:00


## Selecting periods of interest

### Define start/end of period

Start and end periods might be read in from an event log.
For example purposes here, timestamps will be created here.

In [12]:
# Example 10 minutes
START = pd.to_datetime("2025-10-09T14:50:00.000Z").tz_convert(parser.timezone)
END = pd.to_datetime("2025-10-09T15:00:00.000Z").tz_convert(parser.timezone)

# Note: if time string contains 'Z' or a timezone offset, use tz_convert();
#       if no timezone offset, use tz_localize()
# TODO: Add to beatwatch function?

### Select by period

In [13]:
for k, v in data.items():
    v["data_hr"] = select_period(v["data_hr"], START, END) # Replace hr df with selected period
    v["data_accel"] = select_period(v["data_accel"], START, END) # Replace accel df with selected period

2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:select_period — Selecting period from 2025-10-09 10:50:00-04:00 to 2025-10-09 11:00:00-04:00
2026-03-23 12:48:15 | SUCCESS  | beatwatch_process.parsers:_select — DataFrame (single) -> 13280 samples
2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:select_period — Selecting period from 2025-10-09 10:50:00-04:00 to 2025-10-09 11:00:00-04:00
2026-03-23 12:48:15 | SUCCESS  | beatwatch_process.parsers:_select — DataFrame (single) -> 6763 samples
2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:select_period — Selecting period from 2025-10-09 10:50:00-04:00 to 2025-10-09 11:00:00-04:00
2026-03-23 12:48:15 | SUCCESS  | beatwatch_process.parsers:_select — DataFrame (single) -> 12699 samples
2026-03-23 12:48:15 | INFO     | beatwatch_process.parsers:select_period — Selecting period from 2025-10-09 10:50:00-04:00 to 2025-10-09 11:00:00-04:00
2026-03-23 12:48:15 | SUCCESS  | beatwatch_process.parsers:_select — DataFrame

## Preprocess

### IMPORTANT: Uneven sample rate

The bangle.js 2 smartwatch heart rate monitor does not guarantee an exact sample rate.
Before any processing, the samples in each dataframe need to be realigned to an even sample rate.

In [14]:
for k, v in data.items():
    for df_name in ["hr", "accel"]:
        if f"data_{df_name}" in v:
            print(f"Processing {df_name} dataframe [{k}]")
            # Pad selection to nearest second before first sample # TODO: add to beatwath_process
            start = v[f"data_{df_name}"]["time_absolute"].min().floor("s")
            # Resample  data
            v[f"data_{df_name}"] = upsample(
                v[f"data_{df_name}"],
                time_start=start,
                output_rate=cfg["rate_downsample"],
                max_gap=cfg[f"max_gap_{df_name}"],
            )

# TODO: Drop time_elapsed in beatwatch_process

Processing hr dataframe [10-09_time_14-38-58_2cee_W010.csv]
Processing accel dataframe [10-09_time_14-38-58_2cee_W010.csv]


2026-03-23 12:48:16 | WARNING  | beatwatch_process.process:upsample — Missing periods 10485 (>= 312) found in data


Processing hr dataframe [10-09_time_14-38-58_0510_W002.csv]


2026-03-23 12:48:16 | WARNING  | beatwatch_process.process:upsample — Missing periods 113 (>= 160) found in data


Processing accel dataframe [10-09_time_14-38-58_0510_W002.csv]
Processing hr dataframe [10-09_time_14-38-58_7eed_W004.csv]


2026-03-23 12:48:17 | WARNING  | beatwatch_process.process:upsample — Missing periods 76 (>= 160) found in data


Processing accel dataframe [10-09_time_14-38-58_7eed_W004.csv]


2026-03-23 12:48:17 | WARNING  | beatwatch_process.process:upsample — Missing periods 467 (>= 312) found in data


#### New data structure

The time index is now a constant sample rate. Additonally, gaps longer than the configured maximum are labelled.

In [15]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_hr"]

,time_elapsed,heart_rate_bpm,confidence,ppg_raw,ppg_filter,gap
2025-10-09 10:50:00-04:00,NaT,NaN,<NA>,NaN,NaN,<NA>
2025-10-09 10:50:00.040000-04:00,NaT,NaN,<NA>,NaN,NaN,<NA>
2025-10-09 10:50:00.080000-04:00,NaT,NaN,<NA>,NaN,NaN,<NA>
2025-10-09 10:50:00.120000-04:00,0 days 00:11:01.406000,92.0,77.0,7586.000000,-32768.000000,False
2025-10-09 10:50:00.160000-04:00,0 days 00:11:01.446000,92.0,77.0,7594.421053,-32768.000000,False
...,...,...,...,...,...,...
2025-10-09 10:59:59.800000-04:00,0 days 00:21:01.086000,78.0,100.0,2484.600000,-5657.600000,False
2025-10-09 10:59:59.840000-04:00,0 days 00:21:01.126000,78.0,100.0,2505.317073,443.317073,False
2025-10-09 10:59:59.880000-04:00,0 days 00:21:01.166000,78.0,100.0,2638.000000,26572.000000,False
2025-10-09 10:59:59.920000-04:00,0 days 00:21:01.206000,78.0,100.0,1627.268293,-18382.268293,False


In [16]:
data["10-09_time_14-38-58_2cee_W010.csv"]["data_accel"]

,time_elapsed,x,y,z,magnitude,difference,gap
2025-10-09 10:50:00-04:00,NaT,NaN,NaN,NaN,NaN,NaN,<NA>
2025-10-09 10:50:00.040000-04:00,NaT,NaN,NaN,NaN,NaN,NaN,<NA>
2025-10-09 10:50:00.080000-04:00,NaT,NaN,NaN,NaN,NaN,NaN,<NA>
2025-10-09 10:50:00.120000-04:00,0 days 00:11:01.406000,-178.000000,786.333333,-389.333333,897.333333,156.666667,False
2025-10-09 10:50:00.160000-04:00,0 days 00:11:01.446000,-105.619048,800.301587,-401.396825,901.777778,119.841270,False
...,...,...,...,...,...,...,...
2025-10-09 10:59:59.800000-04:00,0 days 00:21:01.086000,101.407407,975.395062,322.000000,1032.197531,4.000000,False
2025-10-09 10:59:59.840000-04:00,0 days 00:21:01.126000,101.700000,976.500000,321.700000,1033.200000,4.300000,False
2025-10-09 10:59:59.880000-04:00,0 days 00:21:01.166000,101.200000,979.000000,321.200000,1035.200000,4.800000,False
2025-10-09 10:59:59.920000-04:00,0 days 00:21:01.206000,100.707317,981.170732,319.536585,1036.585366,5.585366,False


## Storing data

Partially processed datasets can be stored in the `__cache` directory.
The file type is written in `.parquet` format for faster read/write times and 
smaller storage requirements (compared to `.csv`).

### Writing separate data files

Here, an individual file is written for each data frame (e.g., heart rate, acceleration) parsed from each watch.

In [17]:
for k, v in data.items():
    for df_name in ["hr", "accel"]:
        if f"data_{df_name}" in v:
            v[f"data_{df_name}"].to_parquet(
                cfg["paths_out"]["__cache"]
                / f"{k.strip('.csv')}_{df_name}_{hz}Hz.parquet"
            )

### Writing a single data file

If you want data for all watches stored in a single file, first create a single data frame with watch ids:

In [18]:
for df_name in ["hr", "accel"]:
    combined_df = pd.DataFrame()
    for k, v in data.items():
        if f"data_{df_name}" in v:
            # Get watch name
            watch_name = re.search(r"(W.*)\..*$", k).group(1)
            # Add name to dataframe
            v[f"data_{df_name}"]["watch"] = watch_name
            v[f"data_{df_name}"]["watch"] = v[f"data_{df_name}"]["watch"].astype(
                "category"
            ) # FIX: Set type when concat
            # Create single dataframe
            combined_df = pd.concat([combined_df, v[f"data_{df_name}"]])
    # Write
    combined_df["watch"] = combined_df["watch"].astype("category")
    if not combined_df.empty:
        combined_df.to_parquet(
            cfg["paths_out"]["__cache"] / f"data_{df_name}_full_{hz}Hz.parquet"
        )

## Visualisation

### Plot individual data streams

TODO: Add to beatwatch library

In [19]:
subplots = []

for k, v in data.items():
    watch_name = re.search(r"(W.*)\..*$", k).group(1)
    trace_hrbpm = hv.Curve(
        v["data_hr"],
        "index",
        "heart_rate_bpm",
        group=watch_name,
        label="Heart Rate (bpm)",
    ).opts(ylabel="Heart Rate (bpm)", ylim=(0, None), color="black")
    trace_conf = hv.Points(
        v["data_hr"],
        kdims=["index", "heart_rate_bpm"],
        vdims=["confidence"],
        group=watch_name,
        label="Confidence",
    ).opts(
        color="confidence", cmap="bmy", size=5, clabel="Confidence", colorbar=True, ylabel="Heart Rate (bpm)"
    )
    trace_ppg = hv.Curve(
        v["data_hr"], "index", "ppg_raw", group=watch_name, label="PPG Signal"
    ).opts(ylabel="PPG Sensor Value", color="red", alpha=0.6)
    subplots.append(
        hv.Overlay([trace_ppg, trace_conf, trace_hrbpm]).opts(
            xlabel="Time", width=800, height=400, multi_y=True
        )
    )

hv.Layout(subplots).cols(1)

:Layout
   .W010.I :Overlay
      .W010.PPG_Signal                                        :Curve   [index]   (ppg_raw)
      .W010.Confidence                                        :Points   [index,heart_rate_bpm]   (confidence)
      .W010.Heart_Rate_left_parenthesis_bpm_right_parenthesis :Curve   [index]   (heart_rate_bpm)
   .W002.I :Overlay
      .W002.PPG_Signal                                        :Curve   [index]   (ppg_raw)
      .W002.Confidence                                        :Points   [index,heart_rate_bpm]   (confidence)
      .W002.Heart_Rate_left_parenthesis_bpm_right_parenthesis :Curve   [index]   (heart_rate_bpm)
   .W004.I :Overlay
      .W004.PPG_Signal                                        :Curve   [index]   (ppg_raw)
      .W004.Confidence                                        :Points   [index,heart_rate_bpm]   (confidence)
      .W004.Heart_Rate_left_parenthesis_bpm_right_parenthesis :Curve   [index]   (heart_rate_bpm)